# Store Sales — Step 2: Data Preprocessing & Validation Split

**Author:** Bella Ren

Turns the raw Kaggle files into one clean panel table (`processed.parquet`) that every model reads.
Each step below comes from a finding in the EDA notebook.

| # | Step | EDA finding |
|---|---|---|
| 1 | Complete the date grid; flag closed days | Dec 25 missing every year; Jan 1 almost zero |
| 2 | Flag inactive rows before each series' first sale | 8 stores opened after 2013 |
| 3 | Flag zero series (predict 0 directly) | 111 series with zero sales in the last 30 days |
| 4 | Oil price: fill every calendar day | 43 missing values, weekdays only |
| 5 | Holidays matched to each store (national / state / city) | National holidays lift Mon–Thu sales by 25–35% |
| 6 | Calendar features: day of week, payday, earthquake | Strong weekly seasonality, paydays, 2016 earthquake |
| 7 | `log1p(sales)` target | Competition metric is RMSLE |
| 8 | Train / validation / test split (last 16 days = validation) | Test horizon is 16 days |

In [1]:
import pandas as pd
import numpy as np

DATA = 'data/'
VALID_START = pd.Timestamp('2017-07-31')   # last 16 days of train
TEST_START = pd.Timestamp('2017-08-16')

train = pd.read_csv(DATA + 'train.csv', parse_dates=['date'])
test = pd.read_csv(DATA + 'test.csv', parse_dates=['date'])
stores = pd.read_csv(DATA + 'stores.csv')
oil = pd.read_csv(DATA + 'oil.csv', parse_dates=['date'])
hol = pd.read_csv(DATA + 'holidays_events.csv', parse_dates=['date'])
print(train.shape, test.shape)

(3000888, 6) (28512, 5)


## 1. Combine train + test and complete the date grid
Every series gets one row per calendar day. Rows missing from `train` (Dec 25) get `sales = 0`; Dec 25 and Jan 1 are flagged as `closed`.

In [2]:
panel = pd.concat([train, test], ignore_index=True)   # test rows have sales = NaN

dates = pd.date_range(panel.date.min(), panel.date.max())
keys = panel[['store_nbr', 'family']].drop_duplicates()
grid = keys.merge(pd.DataFrame({'date': dates}), how='cross')
df = grid.merge(panel, on=['store_nbr', 'family', 'date'], how='left')

is_train_period = df.date < TEST_START
added = is_train_period & df.sales.isna()
print('Rows added by completing the grid:', added.sum(), '| dates:', sorted(df.loc[added, 'date'].dt.date.unique()))
df.loc[added, ['sales', 'onpromotion']] = 0
df['onpromotion'] = df.onpromotion.fillna(0)

md_ = df.date.dt.strftime('%m-%d')
df['closed'] = md_.isin(['12-25', '01-01']).astype(int)

df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
print(df.shape, '| expected:', len(keys) * len(dates))

Rows added by completing the grid: 7128 | dates: [datetime.date(2013, 12, 25), datetime.date(2014, 12, 25), datetime.date(2015, 12, 25), datetime.date(2016, 12, 25)]


(3036528, 7) | expected: 3036528


## 2. Inactive rows: before each series' first sale
Handles both new stores and product families introduced later in an existing store.
Models should train only on `active == 1` rows. Series that never sold anything are fully inactive.

In [3]:
first_sale = (df[(df.sales > 0) & (df.date < TEST_START)]
              .groupby(['store_nbr', 'family']).date.min().rename('first_sale'))
df = df.merge(first_sale, on=['store_nbr', 'family'], how='left')
df['active'] = (df.date >= df.first_sale).astype(int)
df = df.drop(columns='first_sale')

hist = df[df.date < TEST_START]
print('Inactive share of historical rows: %.1f%%' % (100 * (1 - hist.active.mean())))
print('Series that never sold anything:', first_sale.reindex(pd.MultiIndex.from_frame(keys)).isna().sum())

Inactive share of historical rows: 18.5%
Series that never sold anything: 53


## 3. Zero series
Series with zero total sales in the last 30 days of train. Their final prediction is set to 0.

In [4]:
last30 = df[(df.date > TEST_START - pd.Timedelta(days=31)) & (df.date < TEST_START)]
s30 = last30.groupby(['store_nbr', 'family']).sales.sum()
zero_keys = s30[s30 == 0].index
df['zero_series'] = pd.MultiIndex.from_frame(df[['store_nbr', 'family']]).isin(zero_keys).astype(int)
print('Zero series:', len(zero_keys), '/', len(s30))

Zero series: 111 / 1782


## 4. Oil price
Reindex to every calendar day, interpolate linearly, then fill the edges.

In [5]:
oil_daily = (oil.set_index('date').dcoilwtico
             .reindex(dates).interpolate(method='linear').ffill().bfill()
             .rename('oil').rename_axis('date').reset_index())
print('Missing after filling:', oil_daily.oil.isna().sum())
df = df.merge(oil_daily, on='date', how='left')

Missing after filling: 0


## 5. Holidays matched to each store
Rules:
- `transferred == True` rows are dropped: the holiday was moved, so that date is a normal day. The day it moved to appears as type `Transfer`.
- `Holiday`, `Transfer`, `Additional`, `Bridge` → a day off (`holiday`).
- `Work Day` → a compensating working day (usually a Saturday) (`work_day`).
- `Event` (Mother's Day, Black Friday, Cyber Monday, World Cup, earthquake) → `event`.
- National applies to all stores; Regional matches the store's `state`; Local matches the store's `city`.

In [6]:
h = hol[~hol.transferred].copy()
h['kind'] = np.select(
    [h.type.isin(['Holiday', 'Transfer', 'Additional', 'Bridge']), h.type == 'Work Day', h.type == 'Event'],
    ['holiday', 'work_day', 'event'], default='other')

st = stores[['store_nbr', 'city', 'state']]
nat = h[h.locale == 'National'].merge(st, how='cross')
reg = h[h.locale == 'Regional'].merge(st, left_on='locale_name', right_on='state')
loc = h[h.locale == 'Local'].merge(st, left_on='locale_name', right_on='city')

parts = []
for scope, part in [('national', nat), ('regional', reg), ('local', loc)]:
    part = part[part.kind != 'other'].copy()
    part['col'] = np.where(part.kind == 'holiday', 'hol_' + scope, part.kind)
    parts.append(part[['date', 'store_nbr', 'col']])
hs = pd.concat(parts).drop_duplicates()
hs['v'] = 1
hol_wide = hs.pivot_table(index=['date', 'store_nbr'], columns='col', values='v', aggfunc='max', fill_value=0).reset_index()
hol_wide.columns.name = None

df = df.merge(hol_wide, on=['date', 'store_nbr'], how='left')
hol_cols = ['hol_national', 'hol_regional', 'hol_local', 'work_day', 'event']
for c in hol_cols:
    if c not in df:
        df[c] = 0
df[hol_cols] = df[hol_cols].fillna(0).astype(int)
df['holiday'] = df[['hol_national', 'hol_regional', 'hol_local']].max(axis=1)
print(df[hol_cols + ['holiday']].mean().round(4))

hol_national    0.0481
hol_regional    0.0003
hol_local       0.0039
work_day        0.0029
event           0.0323
holiday         0.0523
dtype: float64


## 6. Calendar features and store attributes

In [7]:
d = df.date.dt
df['dow'] = d.dayofweek
df['dom'] = d.day
df['month'] = d.month
df['year'] = d.year
df['dayofyear'] = d.dayofyear
df['payday'] = ((d.day == 15) | d.is_month_end).astype(int)
df['earthquake'] = df.date.between('2016-04-16', '2016-05-15').astype(int)

df = df.merge(stores.rename(columns={'type': 'store_type'}), on='store_nbr', how='left')

## 7. Target: log1p(sales)

In [8]:
df['log_sales'] = np.log1p(df.sales)

## 8. Train / validation / test split
- `train`: up to 2017-07-30
- `valid`: 2017-07-31 → 2017-08-15 (16 days, same length as the test horizon)
- `test` : 2017-08-16 → 2017-08-31

Final models are refit on train + valid before predicting test.

In [9]:
df['split'] = np.select([df.date >= TEST_START, df.date >= VALID_START], ['test', 'valid'], default='train')
print(df.groupby('split').date.agg(['min', 'max', 'nunique']))

             min        max  nunique
split                               
test  2017-08-16 2017-08-31       16
train 2013-01-01 2017-07-30     1672
valid 2017-07-31 2017-08-15       16


## 9. Checks and save

In [10]:
n_series = df.groupby(['store_nbr', 'family']).ngroups
assert len(df) == n_series * df.date.nunique(), 'grid has duplicates or gaps'
assert df.loc[df.split == 'test', 'id'].notna().sum() == len(test), 'test ids lost'
assert df.loc[df.split != 'test', 'sales'].notna().all(), 'missing sales in history'
feat = ['onpromotion', 'oil', 'closed', 'active', 'zero_series', 'holiday', 'work_day', 'event',
        'dow', 'payday', 'earthquake', 'city', 'state', 'store_type', 'cluster']
assert df[feat].notna().all().all(), 'NaN in features'
print('All checks passed. Shape:', df.shape)

for c in ['family', 'city', 'state', 'store_type', 'split']:
    df[c] = df[c].astype('category')
df.to_parquet(DATA + 'processed.parquet', index=False)
print('Saved to', DATA + 'processed.parquet')
df.head()

All checks passed. Shape: (3036528, 29)


Saved to data/processed.parquet


,store_nbr,family,date,id,sales,onpromotion,closed,active,zero_series,oil,...,year,dayofyear,payday,earthquake,city,state,store_type,cluster,log_sales,split
0,1,AUTOMOTIVE,2013-01-01,0.0,0.0,0.0,1,0,0,93.140000,...,2013,1,0,0,Quito,Pichincha,D,13,0.000000,train
1,1,AUTOMOTIVE,2013-01-02,1782.0,2.0,0.0,0,1,0,93.140000,...,2013,2,0,0,Quito,Pichincha,D,13,1.098612,train
2,1,AUTOMOTIVE,2013-01-03,3564.0,3.0,0.0,0,1,0,92.970000,...,2013,3,0,0,Quito,Pichincha,D,13,1.386294,train
3,1,AUTOMOTIVE,2013-01-04,5346.0,3.0,0.0,0,1,0,93.120000,...,2013,4,0,0,Quito,Pichincha,D,13,1.386294,train
4,1,AUTOMOTIVE,2013-01-05,7128.0,5.0,0.0,0,1,0,93.146667,...,2013,5,0,0,Quito,Pichincha,D,13,1.791759,train


## 10. Sanity check: naive baseline on the validation split
Prediction = mean of the last 14 days before validation, per series; zero series → 0.
Any real model must beat this score.

In [11]:
def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

tr = df[df.split == 'train']
k = ['store_nbr', 'family']
base = tr[tr.date >= VALID_START - pd.Timedelta(days=14)].groupby(k, observed=True).sales.mean().rename('pred')
# Zero series decided from the 30 days BEFORE validation (the zero_series column uses data up to the test start, which would leak here)
zero_val = tr[tr.date >= VALID_START - pd.Timedelta(days=30)].groupby(k, observed=True).sales.sum() == 0
val = df[df.split == 'valid'].merge(base.reset_index(), on=k, how='left')
val.loc[pd.MultiIndex.from_frame(val[k]).isin(zero_val[zero_val].index), 'pred'] = 0
print('Validation RMSLE, naive 14-day mean: %.4f' % rmsle(val.sales, val.pred))

Validation RMSLE, naive 14-day mean: 0.5191
